# StateGraph와 노드·엣지

파이썬 함수 나열을 상태 그래프로 옮기면 실행 순서를 어떻게 읽을 수 있을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 상태와 두 노드 정의

다음 코드에서 상태와 두 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

In [2]:
# 상태와 두 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class FlowState(TypedDict):
    question: str
    query: str
    trace: list[str]

def normalize(state: FlowState) -> dict:
    return {'question': state['question'].strip(), 'trace': ['normalize']}

def make_query(state: FlowState) -> dict:
    query = state['question'].replace('?', '').replace('요', '')
    return {'query': query, 'trace': state['trace'] + ['make_query']}
print('노드 2개 정의 완료')

노드 2개 정의 완료


## 그래프 조립과 컴파일

노드와 엣지를 연결한 뒤 실행 가능한 그래프로 컴파일합니다. 다음 셀에서 이 그래프를 호출합니다.

In [3]:
# 그래프 조립과 컴파일 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder = StateGraph(FlowState)
builder.add_node('normalize', normalize)
builder.add_node('make_query', make_query)
builder.add_edge(START, 'normalize')
builder.add_edge('normalize', 'make_query')
builder.add_edge('make_query', END)
graph = builder.compile()
print('컴파일 완료')

컴파일 완료


## 실행 결과 관찰

실행 결과 관찰 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [4]:
# 실행 결과 관찰 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
result = graph.invoke({'question': '  대출 기간은 며칠인가요?  ', 'query': '', 'trace': []})
print('질문:', result['question'])
print('검색어:', result['query'])
print('실행 흔적:', result['trace'])
assert result['trace'] == ['normalize', 'make_query']

질문: 대출 기간은 며칠인가요?
검색어: 대출 기간은 며칠인가
실행 흔적: ['normalize', 'make_query']


## 질문 하나를 바꿔 비교

질문 하나를 바꿔 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
second = graph.invoke({'question': '  운영 시간은 언제인가요?  ', 'query': '', 'trace': []})
print('첫 검색어:', result['query'])
print('바꾼 검색어:', second['query'])
print('같은 실행 흔적:', second['trace'])
assert second['query'] != result['query']
assert second['trace'] == result['trace']

첫 검색어: 대출 기간은 며칠인가
바꾼 검색어: 운영 시간은 언제인가
같은 실행 흔적: ['normalize', 'make_query']


## Mermaid로 구조 확인

컴파일한 그래프에서 Mermaid 문자열을 생성합니다. 출력에서 노드 이름과 연결 방향을 확인합니다.

In [6]:
# 컴파일된 그래프의 실제 노드와 엣지를 Mermaid 문자열로 확인합니다.
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)
assert 'normalize' in mermaid and 'make_query' in mermaid

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	normalize(normalize)
	make_query(make_query)
	__end__([<p>__end__</p>]):::last
	__start__ --> normalize;
	normalize --> make_query;
	make_query --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

